# ASQA Dataset Preparation

This notebook downloads and prepares the ASQA (Answer Summaries for Questions which are Ambiguous) dataset for RAG training.

ASQA is a long-form question answering dataset focused on ambiguous factoid questions.

## 1. Setup and Installation

In [1]:
# Install required packages
!pip install -q datasets pandas tqdm

In [2]:
import pandas as pd
import json
from datasets import load_dataset
from tqdm.auto import tqdm
import os
from pathlib import Path

c:\Users\Admin\OneDrive - VNU-HCMUS\Documents\GitHub\ragas-evaluation\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Download ASQA Dataset

In [3]:
# Download ASQA dataset from HuggingFace
print("Downloading ASQA dataset from HuggingFace...")
dataset = load_dataset("din0s/asqa")

print(f"\nDataset structure:")
print(dataset)

print(f"\nTrain samples: {len(dataset['train'])}")
print(f"Dev samples: {len(dataset['dev'])}")

Generating dev split: 100%|██████████| 948/948 [00:00<00:00, 48704.67 examples/s]


Dataset structure:
DatasetDict({
    train: Dataset({
        features: ['ambiguous_question', 'qa_pairs', 'wikipages', 'annotations', 'sample_id'],
        num_rows: 4353
    })
    dev: Dataset({
        features: ['ambiguous_question', 'qa_pairs', 'wikipages', 'annotations', 'sample_id'],
        num_rows: 948
    })
})

Train samples: 4353
Dev samples: 948


## 3. Explore Data Structure

In [4]:
# Examine a sample from the training set
sample = dataset['train'][0]

print("Sample keys:")
print(sample.keys())
print("\n" + "="*80)

for key, value in sample.items():
    print(f"\n{key}:")
    if isinstance(value, (list, dict)):
        print(json.dumps(value, indent=2)[:500])  # Show first 500 chars
    else:
        print(value)

Sample keys:
dict_keys(['ambiguous_question', 'qa_pairs', 'wikipages', 'annotations', 'sample_id'])


ambiguous_question:
When does the new bunk'd come out?

qa_pairs:
[
  {
    "context": "No context provided",
    "question": "When does episode 42 of bunk'd come out?",
    "short_answers": [
      "May 24, 2017"
    ],
    "wikipage": null
  },
  {
    "context": "No context provided",
    "question": "When does episode 41 of bunk'd come out?",
    "short_answers": [
      "April 28, 2017"
    ],
    "wikipage": null
  },
  {
    "context": "No context provided",
    "question": "When does episode 40 of bunk'd come out?",
    "short_answers": [
      "April 

wikipages:
[
  {
    "title": "List of Bunk'd episodes",
    "url": "https://en.wikipedia.org/wiki/List%20of%20Bunk%27d%20episodes"
  }
]

annotations:
[
  {
    "knowledge": [
      {
        "content": null,
        "wikipage": "List of Bunk'd episodes"
      }
    ],
    "long_answer": "The new bunk'd episode 41 comes out on 

In [5]:
# Analyze data structure in detail
print("Detailed structure analysis:")
print(f"\nAmbiguous question: {sample.get('ambiguous_question', 'N/A')}")
print(f"\nQA pairs count: {len(sample.get('qa_pairs', []))}")
if sample.get('qa_pairs'):
    print(f"First QA pair: {sample['qa_pairs'][0]}")

print(f"\nWikipages count: {len(sample.get('wikipages', []))}")
if sample.get('wikipages'):
    print(f"First wikipage: {sample['wikipages'][0]}")

print(f"\nAnnotations count: {len(sample.get('annotations', []))}")
if sample.get('annotations'):
    print(f"First annotation keys: {sample['annotations'][0].keys()}")
    print(f"Long answer preview: {sample['annotations'][0].get('long_answer', '')[:200]}...")

Detailed structure analysis:

Ambiguous question: When does the new bunk'd come out?

QA pairs count: 3
First QA pair: {'context': 'No context provided', 'question': "When does episode 42 of bunk'd come out?", 'short_answers': ['May 24, 2017'], 'wikipage': None}

Wikipages count: 1
First wikipage: {'title': "List of Bunk'd episodes", 'url': 'https://en.wikipedia.org/wiki/List%20of%20Bunk%27d%20episodes'}

Annotations count: 1
First annotation keys: dict_keys(['knowledge', 'long_answer'])
Long answer preview: The new bunk'd episode 41 comes out on April 21, 2017, episode 42 comes out on April 28, 2017 and episode 42 is due to come out on May 24, 2017. ...


## 4. Data Transformation to CSV Format

Transform ASQA format to match HotpotQA structure:
- `question`: The ambiguous question
- `answer`: Long-form answer from annotations
- `context`: JSON string with Wikipedia pages (title and text)
- `supporting_facts`: JSON string with relevant Wikipedia page titles

In [6]:
def transform_asqa_to_csv_format(dataset_split):
    """
    Transform ASQA dataset to CSV format compatible with HotpotQA structure.
    
    Note: ASQA wikipages only have titles/URLs, not full text.
    We use the long_answer and qa_pairs context as the document content.
    """
    data = []
    
    for idx, sample in enumerate(tqdm(dataset_split, desc="Processing samples")):
        # Extract question
        question = sample.get('ambiguous_question', '')
        if not question:
            continue
        
        # Extract long-form answer from first annotation
        annotations = sample.get('annotations', [])
        if not annotations:
            continue
        
        long_answer = annotations[0].get('long_answer', '')
        if not long_answer:
            continue
        
        # Extract Wikipedia page titles
        wikipages = sample.get('wikipages', [])
        if not wikipages:
            continue
        
        # Build context from available information
        titles = []
        sentences_list = []
        
        # Add wikipage titles with the long answer as content
        for page in wikipages:
            title = page.get('title', '')
            if title:
                titles.append(title)
                # Use long answer as the document content
                sentences = [s.strip() + '.' for s in long_answer.split('.') if s.strip()]
                sentences_list.append(sentences)
        
        # Also add QA pairs as additional context documents
        qa_pairs = sample.get('qa_pairs', [])
        for i, qa in enumerate(qa_pairs[:3]):  # Limit to 3 QA pairs
            qa_question = qa.get('question', '')
            qa_answers = qa.get('short_answers', [])
            if qa_question and qa_answers:
                titles.append(f"QA_{i+1}")
                qa_sentence = f"{qa_question} {', '.join(qa_answers)}."
                sentences_list.append([qa_sentence])
        
        if not titles:
            continue
        
        context = {
            "title": titles,
            "sentences": sentences_list
        }
        
        # Extract supporting facts
        knowledge = annotations[0].get('knowledge', [])
        supporting_titles = []
        
        for k in knowledge:
            if isinstance(k, dict):
                wiki_title = k.get('wikipage', k.get('title', ''))
                if wiki_title and wiki_title not in supporting_titles:
                    supporting_titles.append(wiki_title)
        
        # If no supporting facts found, use wikipage titles
        if not supporting_titles:
            supporting_titles = [p.get('title', '') for p in wikipages if p.get('title')]
        
        supporting_facts = {
            "title": supporting_titles
        }
        
        # Add to data list
        data.append({
            'id': f"asqa_{idx}",
            'question': question,
            'answer': long_answer,
            'context': str(context),
            'supporting_facts': str(supporting_facts)
        })
    
    return pd.DataFrame(data)

In [7]:
# Transform training set
print("Transforming training set...")
df_train = transform_asqa_to_csv_format(dataset['train'])
print(f"Processed {len(df_train)} training samples")

# Transform dev set
print("\nTransforming dev set...")
df_dev = transform_asqa_to_csv_format(dataset['dev'])
print(f"Processed {len(df_dev)} dev samples")

Transforming training set...


Processing samples: 100%|██████████| 4353/4353 [00:00<00:00, 12144.12it/s]


Processed 4353 training samples

Transforming dev set...


Processing samples: 100%|██████████| 948/948 [00:00<00:00, 8219.06it/s]

Processed 948 dev samples


## 5. Data Quality Check

In [8]:
print("Training set sample:")
print(df_train.head())

print("\nDataset info:")
df_train.info()

if len(df_train) > 0:
    print("\nAnswer length statistics:")
    df_train['answer_length'] = df_train['answer'].str.split().str.len()
    print(df_train['answer_length'].describe())
    
    print("\nSample question and answer:")
    sample_row = df_train.iloc[0]
    print(f"Question: {sample_row['question']}")
    print(f"\nAnswer: {sample_row['answer']}")

Training set sample:
       id                                           question  \
0  asqa_0                 When does the new bunk'd come out?   
1  asqa_1  Who won the 2016 ncaa football national champi...   
2  asqa_2  When was the last time the death penalty was u...   
3  asqa_3  Where will failure of the left ventricle cause...   
4  asqa_4        Who won the war between ethiopia and italy?   

                                              answer  \
0  The new bunk'd episode 41 comes out on April 2...   
1  The 2015 - 2016 season's ncaa national footbal...   
2  The last time the death penalty was used in pa...   
3  "Backward" failure of the left ventricle cause...   
4  The first war between Italy and Ethiopia took ...   

                                             context  \
0  {'title': ["List of Bunk'd episodes", 'QA_1', ...   
1  {'title': ['2015 College Football Playoff Nati...   
2  {'title': ['Capital punishment in Pennsylvania...   
3  {'title': ['Heart failure', 'Q

## 6. Save to CSV Files

In [9]:
# Create output directory
output_dir = Path("../data/asqa")
output_dir.mkdir(parents=True, exist_ok=True)

# Save to CSV
train_path = output_dir / "train.csv"
dev_path = output_dir / "dev.csv"

# Remove temporary column before saving
if 'answer_length' in df_train.columns:
    df_train = df_train.drop('answer_length', axis=1)

df_train.to_csv(train_path, index=False, encoding='utf-8-sig')
df_dev.to_csv(dev_path, index=False, encoding='utf-8-sig')

print(f"✅ Saved training data to {train_path}")
print(f"✅ Saved dev data to {dev_path}")
print(f"\nTrain samples: {len(df_train)}")
print(f"Dev samples: {len(df_dev)}")

✅ Saved training data to ..\data\asqa\train.csv
✅ Saved dev data to ..\data\asqa\dev.csv

Train samples: 4353
Dev samples: 948


## 7. Verification: Load and Test

In [10]:
# Verify saved files can be loaded
import ast

df_test = pd.read_csv(train_path)
print(f"Loaded {len(df_test)} samples from saved CSV")

# Test parsing context
sample_context = ast.literal_eval(df_test.iloc[0]['context'])
print(f"\nContext structure:")
print(f"Titles: {sample_context['title'][:3]}...")
print(f"Number of documents: {len(sample_context['title'])}")

# Test parsing supporting facts
sample_sf = ast.literal_eval(df_test.iloc[0]['supporting_facts'])
print(f"\nSupporting facts:")
print(f"Titles: {sample_sf['title']}")

print("\n✅ Data format verification successful!")

Loaded 4353 samples from saved CSV

Context structure:
Titles: ["List of Bunk'd episodes", 'QA_1', 'QA_2']...
Number of documents: 4

Supporting facts:
Titles: ["List of Bunk'd episodes"]

✅ Data format verification successful!


## Summary

Successfully prepared ASQA dataset:
- Downloaded from HuggingFace (`din0s/asqa`)
- Transformed to HotpotQA-compatible CSV format
- Saved to `data/asqa/train.csv` and `data/asqa/dev.csv`
- Ready for RAG training pipeline